In [0]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "abfss://bronze@ecommercestoragev2.dfs.core.windows.net/merchants"
silver_path = "abfss://silver@ecommercestoragev2.dfs.core.windows.net/merchants"
hlp_path    = f"{silver_path}/merchants_hlp"

In [0]:
silver_df = (
    spark.read.format("parquet").load(bronze_path)
    .select("merchant_id")
    .distinct()
)

In [0]:
hlp_exists = DeltaTable.isDeltaTable(spark, hlp_path)

if hlp_exists:
    hlp_df   = spark.read.format("delta").load(hlp_path)
    max_sk   = hlp_df.agg(F.max("merchant_sk")).collect()[0][0] or 0
    new_merchants_df = silver_df.join(
        hlp_df.select("merchant_id"), on="merchant_id", how="left_anti"
    )
else:
    max_sk           = 0
    new_merchants_df = silver_df

In [0]:
window_spec = Window.orderBy("merchant_id")

new_rows_df = (
    new_merchants_df
    .withColumn("merchant_sk",  F.lit(max_sk) + F.row_number().over(window_spec))
    .withColumn("source",       F.lit("merchants"))
    .withColumn("oper",         F.lit("I"))
    .withColumn("ins_tmstmp",   F.current_timestamp())
    .withColumn("upd_tmstmp",   F.current_timestamp())
    .withColumn("batch_id",     F.lit(0))
    .select("merchant_sk", "merchant_id", "source", "oper",
            "ins_tmstmp", "upd_tmstmp", "batch_id")
)

In [0]:
(
    new_rows_df
    .write
    .format("delta")
    .mode("append")
    .save(hlp_path)
)

print(f"Written {new_rows_df.count()} new rows to {hlp_path}")